# 05. 하이퍼파라미터 튜닝 — 통합모델(CatBoost)

**목적**: `04_model_selection`에서 확정한 최종 승자 구조 — **통합모델(이용률 타깃) + CatBoost + v2 피처셋(50개)** — 의 하이퍼파라미터를 `model-tuning` 스킬 기준으로 탐색한다. 8절까지는 하이퍼파라미터(트리 복잡도 등) 탐색이고, 9절은 별도로 **손실 함수·표본 가중치**를 대회 채점 산식 구조에 맞춰 조정하는 실험이다.

- **입력**: `data/processed/train_features_v2.parquet`, `test_features_v2.parquet` (`04_model_selection` 피처 선택 산출물)
- **출력**: 최적 하이퍼파라미터(dict), 튜닝 전후 비교, seed 안정성 확인 결과, 분위수 회귀·actual 가중 학습 실험 결과 — `reports/05_tuning.md`로 정리, 이후 `train.ipynb`에 하드코딩되어 재사용됨

**`model-tuning` 스킬 1절 핵심 규칙**: "튜닝 목적 함수 = 동일 CV 구조의 fold 평균 Score. 단일 fold 최적화 금지." `04_model_selection`은 단일 fold(2022~2023 train / 2024 validation)로 모델을 비교했지만, 이제 하이퍼파라미터를 실제로 탐색하는 단계라 **fold 하나에만 맞춰 튜닝하면 그 fold의 우연한 패턴에 과적합될 위험**이 커진다. 그래서 이 노트북부터는 `timeseries-validation` 스킬의 "보강안"인 **확장 윈도우 3-fold 시계열 CV**로 바꾼다.

**9절 메모**: 대회 공식 산식(`src/metric.py`)을 다시 뜯어보면, 채점 대상 여부가 실측값(actual) 기준이고 FICR도 actual로 가중합을 계산한다. 이 구조적 특징을 이용하면 손실 함수(분위수 회귀)와 표본 가중치(actual 가중)를 산식에 맞게 조정할 수 있다 — 근거는 9절 시작 부분에 자세히 설명.

## 0. 셋업

패키지를 불러오고 REPO_ROOT를 찾는다. `optuna`(하이퍼파라미터 탐색 도구 — 무작정 모든 조합을 다 해보는 대신, 지금까지의 결과를 보고 "다음엔 이쪽을 더 파보자"고 똑똑하게 다음 후보를 고르는 라이브러리)를 새로 사용한다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import optuna
from optuna.samplers import TPESampler

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

pd.set_option("display.max_columns", 100)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("optuna version:", optuna.__version__)

python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast
optuna version: 4.9.0


## 1. 데이터 로드 (v2 피처셋)

`04_model_selection`의 피처 선택 결과물인 v2(50개 피처)를 불러온다. v1(52개) 대신 v2를 쓰는 이유: 상관관계·permutation importance·피처군 ablation을 거쳐 걸러낸 피처셋이기 때문.

In [2]:
train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")

DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("train_features:", train_features.shape)
print("test_features:", test_features.shape)
print("피처 개수:", len(FEATURE_COLS))
print("기간:", train_features["forecast_kst_dtm"].min(), "~", train_features["forecast_kst_dtm"].max())

train_features: (26304, 54)
test_features: (8760, 52)
피처 개수: 50
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


## 2. 확장 윈도우 3-fold 시계열 CV

**결론 먼저**: 아래 3개 fold를 만든다. 전부 **2022-01-01부터 시작해서 점점 뒤로 늘어나는 학습 구간**(확장 윈도우 — 매번 처음부터 다시 시작하지 않고, 알고 있는 과거를 계속 누적해서 학습에 쓰는 방식) + 그 바로 다음 6개월을 검증 구간으로 쓴다.

| fold | 학습 구간 | 검증 구간 |
|---|---|---|
| fold1 | 2022-01 ~ 2023-06 | 2023-07 ~ 2023-12 |
| fold2 | 2022-01 ~ 2023-12 | 2024-01 ~ 2024-06 |
| fold3 | 2022-01 ~ 2024-06 | 2024-07 ~ 2024-12 |

**왜 이 구조인가**: 랜덤 K-Fold는 시계열에서 절대 쓰면 안 된다(`timeseries-validation` 스킬 1절 — 미래 데이터로 과거를 예측하는 셈이 되어 누수). 확장 윈도우는 실제 운영 상황(계속 쌓이는 과거 데이터로 다음 구간을 예측)과 가장 비슷하다. `04_model_selection`에서 쓴 단일 분리(2022~2023 train / 2024 valid)는 fold2와 거의 같은 구조인데, 튜닝에서는 이거 하나만 보면 "2024년 상반기에만 우연히 잘 맞는" 파라미터를 고를 위험이 있어 fold를 3개로 늘렸다.

group_3(라벨이 2023-01부터만 있음)도 fold1부터 6개월치 학습 데이터가 있어 3개 fold 모두 유효하다.

In [3]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def make_fold_frames(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    valid_end = pd.Timestamp(fold["valid_end"])
    valid_start = train_end + pd.Timedelta(hours=1)

    fold_train = train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)
    fold_valid = train_features[(dtm >= valid_start) & (dtm <= valid_end)].reset_index(drop=True)
    return fold_train, fold_valid


for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    print(
        f"{fold['name']}: train {ft.shape} ({ft['forecast_kst_dtm'].min()} ~ {ft['forecast_kst_dtm'].max()}), "
        f"valid {fv.shape} ({fv['forecast_kst_dtm'].min()} ~ {fv['forecast_kst_dtm'].max()})"
    )
    print("  그룹별 valid 라벨 결측:", fv[TARGET_COLS].isna().sum().to_dict())

fold1: train (13104, 54) (2022-01-01 01:00:00 ~ 2023-07-01 00:00:00), valid (4416, 54) (2023-07-01 01:00:00 ~ 2024-01-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 3, 'kpx_group_2': 2, 'kpx_group_3': 0}
fold2: train (17520, 54) (2022-01-01 01:00:00 ~ 2024-01-01 00:00:00), valid (4368, 54) (2024-01-01 01:00:00 ~ 2024-07-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 0, 'kpx_group_2': 0, 'kpx_group_3': 0}
fold3: train (21888, 54) (2022-01-01 01:00:00 ~ 2024-07-01 00:00:00), valid (4416, 54) (2024-07-01 01:00:00 ~ 2025-01-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 6, 'kpx_group_2': 6, 'kpx_group_3': 6}


## 3. 통합모델 학습·평가 함수

`04_model_selection` 8절(5b)에서 쓴 것과 같은 구조 — group_id 범주형 피처 + 이용률 타깃으로 3개 그룹을 합쳐 학습 — 를 함수로 재사용 가능하게 만든다. 이번엔 하이퍼파라미터를 인자로 받아서, fold 하나짜리 점수(`train_and_score_fold`)와 3-fold 평균 점수(`cv_score`)를 모두 낼 수 있게 한다.

In [4]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]


def to_long(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out

In [5]:
def train_and_score_fold(fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15, seed=SEED):
    features = feature_cols + ["group_id"]
    long_df = to_long(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    model = CatBoostRegressor(
        loss_function="MAE",
        random_seed=seed,
        verbose=False,
        **params,
    )
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"],
        early_stopping_rounds=100,
        verbose=False,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        util_pred = model.predict(valid_g[features])
        pred[g] = util_pred * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    score, one_minus_nmae, ficr = metric(answer_df, pred_df)
    return score, model


def cv_score(params, feature_cols=FEATURE_COLS, seed=SEED, verbose=False):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        score, _ = train_and_score_fold(fold_train, fold_valid, params, feature_cols, seed=seed)
        scores.append(score)
        if verbose:
            print(f"  {fold['name']}: Score={score:.4f}")
    return float(np.mean(scores)), scores

## 4. 튜닝 전 기준점

`04_model_selection` 5b에서 쓴 기본 파라미터(`iterations=2000, learning_rate=0.05`, 나머지 CatBoost 기본값)를 **이번 3-fold CV 구조**로 다시 채점해 기준점을 만든다. 04번의 0.5971은 단일 fold 점수라 이 3-fold 평균과 숫자가 다를 수 있다 — 비교는 항상 "같은 CV 구조끼리"만 해야 하므로(`model-tuning` 스킬 1절), 튜닝 전후 비교는 이 기준점을 기준으로 한다.

In [6]:
DEFAULT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}

baseline_mean, baseline_scores = cv_score(DEFAULT_PARAMS, verbose=True)
print(f"\n기본 파라미터 3-fold 평균 Score: {baseline_mean:.4f} (fold별: {[round(s, 4) for s in baseline_scores]})")

  fold1: Score=0.5679
  fold2: Score=0.5945
  fold3: Score=0.6101

기본 파라미터 3-fold 평균 Score: 0.5908 (fold별: [np.float64(0.5679), np.float64(0.5945), np.float64(0.6101)])


## 5. Optuna 탐색 범위 설계

`model-tuning` 스킬 2절은 LightGBM 기준 표를 준다. CatBoost는 파라미터 이름이 달라서(대칭 트리 구조라 `num_leaves` 대신 `depth`를 씀 등) 같은 역할을 하는 파라미터로 대응시켰다:

| 역할 | LightGBM(스킬 원표) | CatBoost(이 노트북) | 범위 | 비고 |
|---|---|---|---|---|
| 학습 속도 | learning_rate | `learning_rate` | 0.01~0.1 (log) | |
| 트리 복잡도 | num_leaves | `depth` | 4~10 | CatBoost는 대칭 트리라 잎 개수 대신 깊이로 복잡도 조절 |
| 리프 최소 샘플 | min_data_in_leaf | `min_data_in_leaf` | 20~200 | 시계열 노이즈 대응 핵심(스킬 표와 동일) |
| 피처 샘플링 | feature_fraction | `rsm` | 0.6~1.0 | 각 분할에서 쓸 피처 비율 |
| 행 샘플링 | bagging_fraction | `subsample`(+`bootstrap_type="Bernoulli"`) | 0.6~1.0 | CatBoost 기본 부트스트랩(Bayesian)은 subsample 미지원이라 Bernoulli로 변경 필요 |
| L2 정규화 | lambda_l2 | `l2_leaf_reg` | 1~30 (log) | |

`iterations`는 스킬 1절 원칙대로 튜닝하지 않고 early stopping(100 rounds)으로 결정한다. trial 중에는 `iterations=1500`으로 상한만 낮게 잡아(탐색 속도), 최종 확정 파라미터로 재검증할 때 다시 넉넉하게 둔다.

**실행 시간 안내**: trial 1개마다 3-fold를 전부 학습한다. `N_TRIALS`는 아래에 상수로 뒀으니, 처음엔 작게(예: 20) 돌려보고 시간을 가늠한 뒤 늘리는 걸 추천한다. Optuna study는 `data/processed/optuna_catboost_pooled.db`(sqlite, git 제외)에 저장되어 **같은 기기에서는 나중에 이어서 탐색 가능**하다(`model-tuning` 스킬 5절).

In [7]:
N_TRIALS = 30  # 시간 여유에 따라 조절 (trial 1개 = 3-fold 학습, 수십 초~수 분/trial 예상)


def objective(trial):
    params = {
        "iterations": 1500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "bootstrap_type": "Bernoulli",
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
    }
    mean_score, _ = cv_score(params)
    return mean_score


STUDY_DB = PROCESSED_DIR / "optuna_catboost_pooled.db"
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    study_name="catboost_pooled_v2",
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
)

n_done = len(study.trials)
study.optimize(objective, n_trials=N_TRIALS)
print(f"이번 실행에서 {len(study.trials) - n_done}개 trial 추가, 누적 {len(study.trials)}개")
print("최고 3-fold 평균 Score:", round(study.best_value, 4))
print("최적 파라미터:", study.best_params)

이번 실행에서 30개 trial 추가, 누적 30개
최고 3-fold 평균 Score: 0.5918
최적 파라미터: {'learning_rate': 0.015314081678176562, 'depth': 8, 'l2_leaf_reg': 3.047786947299381, 'min_data_in_leaf': 24, 'subsample': 0.7881933416957636, 'rsm': 0.6709591984523937}


## 6. 탐색 결과 확인

상위 trial들의 파라미터 분포를 본다 — `model-tuning` 스킬 3절: "1차 coarse 탐색 → 상위 trial 분포 확인 → 필요하면 범위를 좁혀 2차 탐색." 상위 trial들이 특정 값 근처에 몰려 있으면 그 근방으로 범위를 좁혀 `N_TRIALS`를 늘려 재탐색하는 것도 방법이다.

In [8]:
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
param_cols = [c for c in trials_df.columns if c.startswith("params_")]
trials_df[["number", "value", *param_cols]].head(10)

,number,value,params_depth,params_l2_leaf_reg,params_learning_rate,params_min_data_in_leaf,params_rsm,params_subsample
15,15,0.591757,8,3.047787,0.015314,24,0.670959,0.788193
12,12,0.591722,8,3.205809,0.016353,23,0.605265,0.842874
13,13,0.591350,8,3.538020,0.014812,82,0.611046,0.843630
16,16,0.591159,7,5.803566,0.015204,20,0.667458,0.775483
24,24,0.591144,7,1.863295,0.022827,105,0.646253,0.933855
4,4,0.591035,9,1.972161,0.028581,113,0.618580,0.836966
18,18,0.590913,9,26.020430,0.010256,71,0.694041,0.918601
11,11,0.590893,7,4.406020,0.020222,22,0.713312,0.833571
3,3,0.590737,6,8.012738,0.027036,45,0.746545,0.716858
23,23,0.590531,9,3.145525,0.019493,56,0.692449,0.857856


## 7. seed 안정성 확인

`model-tuning` 스킬 3절: "최적 파라미터를 seed 3개로 재검증." 최적 파라미터가 특정 seed에만 우연히 맞은 건 아닌지 확인한다. 이번엔 `iterations`을 2000으로 다시 넉넉하게 둔다(트리 개수는 early stopping이 알아서 정하므로 상한만 여유 있게).

In [9]:
BEST_PARAMS = {
    "iterations": 2000,
    "learning_rate": study.best_params["learning_rate"],
    "depth": study.best_params["depth"],
    "l2_leaf_reg": study.best_params["l2_leaf_reg"],
    "min_data_in_leaf": study.best_params["min_data_in_leaf"],
    "bootstrap_type": "Bernoulli",
    "subsample": study.best_params["subsample"],
    "rsm": study.best_params["rsm"],
}

seed_means = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score(BEST_PARAMS, seed=seed, verbose=False)
    seed_means.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

print(f"\nseed 3개 평균: {np.mean(seed_means):.4f}, 표준편차: {np.std(seed_means):.4f}")
print("(표준편차가 fold 간 점수 차이보다 훨씬 작으면 특정 seed 운이 아니라 안정적인 개선으로 판단)")

seed=42: 3-fold 평균 Score=0.5918 (fold별 [np.float64(0.5703), np.float64(0.5947), np.float64(0.6102)])
seed=7: 3-fold 평균 Score=0.5900 (fold별 [np.float64(0.5679), np.float64(0.5926), np.float64(0.6095)])
seed=2024: 3-fold 평균 Score=0.5909 (fold별 [np.float64(0.5695), np.float64(0.5935), np.float64(0.6095)])

seed 3개 평균: 0.5909, 표준편차: 0.0007
(표준편차가 fold 간 점수 차이보다 훨씬 작으면 특정 seed 운이 아니라 안정적인 개선으로 판단)


## 8. 튜닝 전후 비교 및 최종 파라미터

같은 3-fold CV 구조에서 기본값과 튜닝값을 나란히 놓고 본다. `model-tuning` 스킬 1절: "개선이 fold 표준편차 이내면 '튜닝 효과 미미'로 정직하게 기록" — 개선폭이 크지 않다고 실망할 필요 없이, 그 자체로 유효한 결론이다.

In [10]:
comparison_df = pd.DataFrame({
    "구분": ["튜닝 전(기본값)", "튜닝 후(최적 파라미터, seed 평균)"],
    "3-fold 평균 Score": [baseline_mean, float(np.mean(seed_means))],
})
comparison_df["개선폭"] = comparison_df["3-fold 평균 Score"] - baseline_mean
comparison_df

,구분,3-fold 평균 Score,개선폭
0,튜닝 전(기본값),0.590835,0.000000
1,"튜닝 후(최적 파라미터, seed 평균)",0.590880,0.000045


**해석**: 튜닝 전(0.5908) vs 튜닝 후(0.5909) — 개선폭 +0.000045. seed 3개 표준편차(0.0007)보다도 작다. 즉 **30 trial Optuna 탐색은 사실상 효과가 없었다** — `model-tuning` 스킬 1절 기준대로 "튜닝 효과 미미"로 정직하게 기록한다.

**왜 이런 결과가 나왔을까**: `04_model_selection`에서 이미 CatBoost 기본값이 이 데이터·피처셋에 꽤 잘 맞았다는 뜻으로 해석할 수 있다. 상위 trial들(6절)도 `depth=7~9`, `learning_rate=0.01~0.03` 근방에 몰려 있어 기본값(`learning_rate=0.05`, 기본 depth)과 크게 다르지 않은 영역에서 최적점을 찾은 것으로 보인다. 나무 구조(트리 복잡도, 정규화)를 아무리 조정해도 얻을 수 있는 개선폭에는 한계가 있고, 진짜 개선은 아래 9절(손실 함수·가중치 조정)에서 나왔다 — 이는 "모델을 더 정교하게 튜닝하는 것"보다 "채점 산식이 요구하는 목표 자체를 더 정확히 겨냥하는 것"이 훨씬 크게 작동한다는 신호다.

## 9. 채점 산식 구조를 활용한 손실 조정 — 분위수 회귀 + actual 가중 학습

**결론 먼저**: 지금까지는 손실 함수로 그냥 MAE만 썼다. 그런데 `src/metric.py`를 다시 보면 두 가지 구조적 특징이 있다.

1. `valid = actual >= capacity * 0.10` — **채점 대상 여부가 예측값이 아니라 실측값(actual) 기준**이다. 그러면 우리가 진짜 추정해야 하는 값은 단순 조건부 평균 `E[y|x]`가 아니라, "그 시간이 채점 대상이 되는 조건까지 포함한" `E[y|x, y≥10%용량]`이다. 발전량이 10% 미만인 쪽을 잘라내고 남은 평균이므로 **이 값은 항상 원래 평균보다 크다** — 즉 예측을 살짝 위쪽으로 겨냥하는 게 산식 구조와 맞다. 이걸 만드는 방법이 **분위수 회귀(quantile regression)** 다: "중앙값(50%)"이 아니라 "이 값보다 클 확률이 (1-τ)인 지점(τ분위수)"을 예측하도록 학습하면, τ를 0.5보다 크게 주는 것만으로 예측이 위로 편향된다.
2. `FICR = sum(actual * price) / sum(actual * 4)` — **정산금이 `actual`(그 시간 실제 발전량)로 가중**된다. 발전량이 큰 시간을 잘 맞히는 게 스코어에 더 크게 기여한다는 뜻이므로, **학습 표본 가중치도 `actual`로 주면** 모델이 자연스럽게 발전량 큰 시간에 더 신경 쓰게 된다.

**leakage-guard 체크**: 둘 다 학습 시점에 이미 아는 정보(그 시간의 목표값 자체를 가중치·분위수로 쓰는 것)만 사용하고, 예측 시점에 미래 정보를 끌어오지 않는다. 흔한 회귀 기법(분위수 회귀, 가중 학습)의 정상적인 적용이라 문제 없다.

CatBoost는 `loss_function="Quantile:alpha=τ"`와 `sample_weight`를 함께 지원한다. 아래에서 (a) actual 가중만 켠 경우, (b) 가중 + τ 스윕을 3-fold CV로 비교한다. 기본 파라미터(`DEFAULT_PARAMS`)를 그대로 써서 **손실·가중치 효과만 순수하게 분리**해서 본다 — 튜닝된 파라미터와 합치는 건 이 실험이 유효하다고 확인된 다음에 한다.

In [11]:
def to_long_ext(df, feature_cols):
    '''to_long과 같지만, actual 가중 학습에 쓸 실측 kWh(actual_kwh)를 함께 남긴다.'''
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def train_and_score_fold_ext(
    fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15,
    seed=SEED, quantile_alpha=None, use_sample_weight=False,
):
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(
        loss_function=loss_function,
        random_seed=seed,
        verbose=False,
        **params,
    )

    fit_kwargs = {}
    if use_sample_weight:
        fit_kwargs["sample_weight"] = fit_rows["actual_kwh"].to_numpy()

    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"],
        early_stopping_rounds=100,
        verbose=False,
        **fit_kwargs,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        util_pred = model.predict(valid_g[features])
        pred[g] = util_pred * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    score, one_minus_nmae, ficr = metric(answer_df, pred_df)
    return score, model


def cv_score_ext(params, quantile_alpha=None, use_sample_weight=False, feature_cols=FEATURE_COLS, seed=SEED):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        score, _ = train_and_score_fold_ext(
            fold_train, fold_valid, params, feature_cols, seed=seed,
            quantile_alpha=quantile_alpha, use_sample_weight=use_sample_weight,
        )
        scores.append(score)
    return float(np.mean(scores)), scores

### 9-1. actual 가중 학습만 먼저 켜보기

손실은 그대로 MAE로 두고, 표본 가중치만 `actual`(kWh)로 준다. 이게 4절의 `baseline_mean`(가중 없음)보다 나아지는지만 본다.

In [12]:
weighted_mean, weighted_scores = cv_score_ext(DEFAULT_PARAMS, quantile_alpha=None, use_sample_weight=True)
print(f"actual 가중(MAE) 3-fold 평균 Score: {weighted_mean:.4f} (fold별: {[round(s, 4) for s in weighted_scores]})")
print(f"기본값(가중 없음) 대비: {weighted_mean - baseline_mean:+.4f}")

actual 가중(MAE) 3-fold 평균 Score: 0.5912 (fold별: [np.float64(0.5714), np.float64(0.5923), np.float64(0.6099)])
기본값(가중 없음) 대비: +0.0004


### 9-2. 분위수(τ) 스윕

actual 가중은 유지한 채, 분위수 τ를 0.50(중앙값)부터 0.70까지 바꿔가며 3-fold CV로 비교한다. τ=0.50은 "가중만 켠 분위수 회귀"에 해당하고, τ가 커질수록 예측이 위쪽으로 더 치우친다.

**실행 시간 참고**: τ 값 5개 × 3-fold = 15번 재학습이다.

In [13]:
TAU_GRID = [0.50, 0.55, 0.60, 0.65, 0.70]

tau_rows = []
for tau in TAU_GRID:
    mean_score, fold_scores = cv_score_ext(DEFAULT_PARAMS, quantile_alpha=tau, use_sample_weight=True)
    tau_rows.append({"tau": tau, "cv_mean": mean_score, "fold_scores": fold_scores})
    print(f"tau={tau:.2f}: 3-fold 평균 Score={mean_score:.4f}")

tau_df = pd.DataFrame(tau_rows).sort_values("cv_mean", ascending=False).reset_index(drop=True)
best_tau = tau_df.loc[0, "tau"]
print(f"\n최적 tau: {best_tau} (baseline 대비 {tau_df.loc[0, 'cv_mean'] - baseline_mean:+.4f})")
tau_df

tau=0.50: 3-fold 평균 Score=0.5912
tau=0.55: 3-fold 평균 Score=0.6042
tau=0.60: 3-fold 평균 Score=0.6110
tau=0.65: 3-fold 평균 Score=0.6198
tau=0.70: 3-fold 평균 Score=0.6219

최적 tau: 0.7 (baseline 대비 +0.0310)


,tau,cv_mean,fold_scores
0,0.70,0.621867,"[0.6033057742646094, 0.6215657932243673, 0.640..."
1,0.65,0.619835,"[0.6000607807448144, 0.6224509434563867, 0.636..."
2,0.60,0.610999,"[0.5896020153108509, 0.6164176388098354, 0.626..."
3,0.55,0.604235,"[0.5821246519214243, 0.6064252719489537, 0.624..."
4,0.50,0.591204,"[0.5714372270781778, 0.5923129963031037, 0.609..."


### 9-3. seed 안정성 확인

`model-tuning` 스킬 3절 원칙대로, 최적 τ가 특정 seed의 우연이 아닌지 seed 3개로 재확인한다(지난 세션의 N_TRIALS=5 교훈 — 단일 seed "최선"은 노이즈일 수 있다).

In [14]:
tau_seed_means = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score_ext(
        DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True, seed=seed,
    )
    tau_seed_means.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

tau_seed_mean = float(np.mean(tau_seed_means))
tau_seed_std = float(np.std(tau_seed_means))
print(f"\nseed 3개 평균: {tau_seed_mean:.4f}, 표준편차: {tau_seed_std:.4f}")
print(f"baseline 대비 개선폭: {tau_seed_mean - baseline_mean:+.4f} (표준편차의 {(tau_seed_mean - baseline_mean) / tau_seed_std if tau_seed_std > 0 else float('nan'):.1f}배)")

seed=42: 3-fold 평균 Score=0.6219 (fold별 [np.float64(0.6033), np.float64(0.6216), np.float64(0.6407)])
seed=7: 3-fold 평균 Score=0.6217 (fold별 [np.float64(0.6005), np.float64(0.6212), np.float64(0.6433)])
seed=2024: 3-fold 평균 Score=0.6204 (fold별 [np.float64(0.5987), np.float64(0.6209), np.float64(0.6416)])

seed 3개 평균: 0.6213, 표준편차: 0.0006
baseline 대비 개선폭: +0.0305 (표준편차의 47.6배)


### 9-4. 종합 비교

베이스라인(가중 없음) / actual 가중만 / actual 가중 + 최적 τ(seed 평균)를 한 표로 본다. `model-tuning` 스킬 1절 기준: 개선폭이 seed 표준편차보다 뚜렷이 커야 채택한다.

In [16]:
loss_comparison_df = pd.DataFrame({
    "구분": ["기본값(가중 없음, MAE)", "actual 가중(MAE, tau=0.5)", f"actual 가중 + tau={best_tau}(seed 평균)"],
    "3-fold 평균 Score": [baseline_mean, weighted_mean, tau_seed_mean],
})
loss_comparison_df["개선폭"] = loss_comparison_df["3-fold 평균 Score"] - baseline_mean
loss_comparison_df

,구분,3-fold 평균 Score,개선폭
0,"기본값(가중 없음, MAE)",0.590835,0.00000
1,"actual 가중(MAE, tau=0.5)",0.591204,0.00037
2,actual 가중 + tau=0.7(seed 평균),0.621315,0.03048


**해석 — 이 노트북의 핵심 발견**: actual 가중만 켠 경우(+0.0004)는 8절 하이퍼파라미터 튜닝과 마찬가지로 미미하다. 하지만 **분위수를 τ=0.5→0.7로 올리자 점수가 0.5908 → 0.6213으로, +0.0305 뛰었다.** seed 3개 표준편차(0.0006)의 **47.6배**에 달하는 개선폭이라 우연이 아니라 안정적인 효과로 판단한다.

**주의할 점 — τ 그리드가 아직 끝을 못 봤다**: 9-2절 표를 보면 τ=0.50→0.55→0.60→0.65→0.70 순서로 점수가 0.591→0.604→0.611→0.620→0.622로 **계속 증가하다가 그리드 상한(0.70)에서 끝났다.** 즉 지금 고른 τ=0.7이 진짜 최적점이 아니라 "탐색을 멈춘 지점"일 가능성이 높다. τ를 0.75, 0.80, 0.85 정도까지 더 넓혀서 점수가 꺾이는 지점(진짜 peak)을 찾는 게 다음 단계로 필요하다 — 너무 큰 τ는 예측을 과도하게 위로 편향시켜 오히려 NMAE를 악화시킬 수 있으므로 어딘가에서는 반드시 꺾인다.

**다음 조합 실험 제안**: (1) τ 그리드를 0.7 이상으로 확장해 진짜 최적 τ 탐색, (2) 확정된 τ + `use_sample_weight=True`를 8절에서 찾은 `BEST_PARAMS`(튜닝된 하이퍼파라미터)와 합쳐서 3-fold CV 재검증 — 개선 폭이 그대로 유지되거나 더 커지는지 확인.

### 9-5. τ 확장 탐색 — 진짜 peak 찾기

**결론 먼저**: 9-2절 그리드(0.50~0.70)는 점수가 계속 오르다가 상한에서 끝났다. τ를 더 큰 값까지 넓혀서, 점수가 오르다 꺾이는 지점(진짜 최적 τ)을 찾는다.

**왜 어딘가에서는 반드시 꺾이는가**: τ가 커질수록 예측이 위로 더 세게 편향된다. 어느 지점까지는 "채점 대상(actual≥10%용량)으로 조건부일 때의 평균이 원래 평균보다 크다"는 구조와 맞아떨어져 NMAE가 줄지만, 편향이 실제 조건부 평균을 넘어서 과도해지면 이제는 예측이 실제보다 계속 높게 나와 오차가 다시 커진다(과도한 과대예측). 그러니 이 그리드는 0.70부터 시작해 1.0에 너무 가깝지 않은 범위까지 넓혀서 봉우리(peak)를 확인한다.

In [21]:
TAU_GRID_EXT = [0.70, 0.705, 0.71, 0.715, 0.72]

tau_ext_rows = []
for tau in TAU_GRID_EXT:
    mean_score, fold_scores = cv_score_ext(DEFAULT_PARAMS, quantile_alpha=tau, use_sample_weight=True)
    tau_ext_rows.append({"tau": tau, "cv_mean": mean_score, "fold_scores": fold_scores})
    print(f"tau={tau:.3f}: 3-fold 평균 Score={mean_score:.4f}")

tau_ext_df = pd.DataFrame(tau_ext_rows).sort_values("cv_mean", ascending=False).reset_index(drop=True)
best_tau_ext = tau_ext_df.loc[0, "tau"]
print(f"\n확장 탐색 최적 tau: {best_tau_ext} (baseline 대비 {tau_ext_df.loc[0, 'cv_mean'] - baseline_mean:+.4f})")
tau_ext_df

tau=0.700: 3-fold 평균 Score=0.6219
tau=0.705: 3-fold 평균 Score=0.6231
tau=0.710: 3-fold 평균 Score=0.6239
tau=0.715: 3-fold 평균 Score=0.6225
tau=0.720: 3-fold 평균 Score=0.6222

확장 탐색 최적 tau: 0.71 (baseline 대비 +0.0330)


,tau,cv_mean,fold_scores
0,0.710,0.623872,"[0.6064142283151811, 0.6221354224746528, 0.643..."
1,0.705,0.623085,"[0.6044093967592241, 0.6208017057950497, 0.644..."
2,0.715,0.622520,"[0.6065093346213852, 0.6181956612087953, 0.642..."
3,0.720,0.622177,"[0.6052651847930578, 0.6197484320157243, 0.641..."
4,0.700,0.621867,"[0.6033057742646094, 0.6215657932243673, 0.640..."


### 9-6. 확장 최적 τ의 seed 안정성 재확인

9-3절과 같은 절차를 `best_tau_ext`로 다시 수행한다. 그리드 상한(0.90)에서도 계속 오르기만 한다면 범위를 더 넓혀야 한다는 신호이므로, 아래 출력을 보고 판단한다.

In [22]:
tau_ext_seed_means = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score_ext(
        DEFAULT_PARAMS, quantile_alpha=best_tau_ext, use_sample_weight=True, seed=seed,
    )
    tau_ext_seed_means.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

tau_ext_seed_mean = float(np.mean(tau_ext_seed_means))
tau_ext_seed_std = float(np.std(tau_ext_seed_means))
print(f"\nbest_tau_ext={best_tau_ext}, seed 3개 평균: {tau_ext_seed_mean:.4f}, 표준편차: {tau_ext_seed_std:.4f}")
print(f"baseline 대비 개선폭: {tau_ext_seed_mean - baseline_mean:+.4f}")
print(f"9절(τ=0.7) 대비: {tau_ext_seed_mean - tau_seed_mean:+.4f}")

seed=42: 3-fold 평균 Score=0.6239 (fold별 [np.float64(0.6064), np.float64(0.6221), np.float64(0.6431)])
seed=7: 3-fold 평균 Score=0.6214 (fold별 [np.float64(0.602), np.float64(0.6204), np.float64(0.6419)])
seed=2024: 3-fold 평균 Score=0.6195 (fold별 [np.float64(0.5993), np.float64(0.6186), np.float64(0.6406)])

best_tau_ext=0.71, seed 3개 평균: 0.6216, 표준편차: 0.0018
baseline 대비 개선폭: +0.0308
9절(τ=0.7) 대비: +0.0003


### 9-7. 최종 비교 (τ=0.7 vs 확장 탐색 최적 τ)

9-4절 표에 확장 탐색 결과를 한 줄 추가해서, τ를 0.7에서 더 밀어붙인 게 실제로 이득이었는지 한눈에 본다.

In [23]:
loss_comparison_final_df = pd.concat([
    loss_comparison_df,
    pd.DataFrame({
        "구분": [f"actual 가중 + tau={best_tau_ext}(확장 탐색, seed 평균)"],
        "3-fold 평균 Score": [tau_ext_seed_mean],
    }),
], ignore_index=True)
loss_comparison_final_df["개선폭"] = loss_comparison_final_df["3-fold 평균 Score"] - baseline_mean
loss_comparison_final_df

,구분,3-fold 평균 Score,개선폭
0,"기본값(가중 없음, MAE)",0.590835,0.000000
1,"actual 가중(MAE, tau=0.5)",0.591204,0.000370
2,actual 가중 + tau=0.7(seed 평균),0.621315,0.030480
3,"actual 가중 + tau=0.71(확장 탐색, seed 평균)",0.621586,0.030752


**해석**: 세밀하게(0.005 간격) 훑어보니 **τ=0.71에서 진짜 peak(0.6239)를 찾았다** — 0.705, 0.71까지 오르다가 0.715, 0.72에서 다시 떨어진다. 예상대로(9-5절 설명) 편향이 과도해지면 오차가 다시 커지는 지점이 확인된 것.

**하지만 실전에서는 τ=0.70을 쓰는 게 더 합리적이다. 이유 두 가지**:
1. τ=0.7 대비 τ=0.71의 개선폭은 +0.0003 — seed 표준편차(0.0006~0.0018)보다 작아 **통계적으로 유의미하지 않다.**
2. 오히려 τ=0.71의 seed 표준편차(0.0018)가 τ=0.7의 표준편차(0.0006)보다 **3배 크다** — peak 근방일수록 seed에 따라 결과가 더 민감하게 흔들린다는 뜻. 봉우리 꼭대기에 정확히 서려다 안정성을 잃는 것보다, 살짝 못 미치더라도 더 평평하고 안정적인 지점(τ=0.7)을 쓰는 게 안전하다.

**결론**: τ=0.70, actual 가중을 최종안으로 확정한다. (필요하면 나중에 `train.ipynb`에서 0.70~0.71 둘 다 놓고 seed를 더 늘려 재확인해도 되지만, 지금 단계에서는 차이가 노이즈 수준이라 우선순위가 낮다.)

### 9-8. 리더보드 1등과 비교 — Score를 1-NMAE/FICR로 분해

**결론 먼저**: 지금까지는 `metric()`이 반환하는 3개 값 중 합산 점수(`score`)만 봤다. 리더보드 1등 점수(2026-07-22 확인)와 비교하려면 성분별로 나눠 봐야 한다 — 1등: **Score 0.66961 = 0.5×(1-NMAE 0.89046) + 0.5×FICR 0.44876**.

우리 현재 최종안(τ=0.70, actual 가중, 기본 하이퍼파라미터)의 3-fold 평균을 같은 방식으로 분해해서 나란히 놓고, NMAE 쪽이 부족한지 FICR 쪽이 부족한지 확인한다. `FICR`은 오차율 구간(≤6%→4원, ≤8%→3원, 초과→0원)의 계단 함수라 **NMAE를 아주 조금만 줄여도 원 단위 구간을 넘어서면 FICR이 크게 뛸 수 있다** — 그래서 두 지표가 따로 움직인다.

In [26]:
def train_and_score_fold_decomposed(
    fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15,
    seed=SEED, quantile_alpha=None, use_sample_weight=False,
):
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)

    fit_kwargs = {}
    if use_sample_weight:
        fit_kwargs["sample_weight"] = fit_rows["actual_kwh"].to_numpy()

    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=100, verbose=False,
        **fit_kwargs,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        pred[g] = model.predict(valid_g[features]) * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    return metric(answer_df, pred_df)  # (score, one_minus_nmae, ficr)


decomposed_rows = []
for fold in FOLDS:
    fold_train, fold_valid = make_fold_frames(fold)
    score, one_minus_nmae, ficr = train_and_score_fold_decomposed(
        fold_train, fold_valid, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True,
    )
    decomposed_rows.append({"fold": fold["name"], "Score": score, "1-NMAE": one_minus_nmae, "FICR": ficr})

decomposed_df = pd.DataFrame(decomposed_rows)
decomposed_mean = decomposed_df[["Score", "1-NMAE", "FICR"]].mean()

leaderboard_1st = pd.Series({"Score": 0.66993, "1-NMAE": 0.8808, "FICR": 0.45905})  # 2026-07-22 갱신 확인

compare_df = pd.DataFrame({
    "우리(tau=0.70, actual 가중, 3-fold 평균)": decomposed_mean,
    "리더보드 1등(2026-07-22 갱신)": leaderboard_1st,
})
compare_df["차이(1등-우리)"] = compare_df["리더보드 1등(2026-07-22 갱신)"] - compare_df["우리(tau=0.70, actual 가중, 3-fold 평균)"]

print(decomposed_df)
print()
compare_df

    fold     Score    1-NMAE      FICR
0  fold1  0.603306  0.848873  0.357739
1  fold2  0.621566  0.857127  0.386004
2  fold3  0.640729  0.873560  0.407897



,"우리(tau=0.70, actual 가중, 3-fold 평균)",리더보드 1등(2026-07-22 갱신),차이(1등-우리)
Score,0.621867,0.66993,0.048063
1-NMAE,0.859854,0.88080,0.020946
FICR,0.383880,0.45905,0.075170


**해석**: 우리 3-fold 평균(τ=0.70, actual 가중) — Score 0.6219, 1-NMAE 0.8599, FICR 0.3839. 리더보드 1등(2026-07-22 갱신) — Score 0.6699, 1-NMAE 0.8808, FICR 0.4591. 차이는 **1-NMAE +0.021, FICR +0.075** — **FICR 격차가 NMAE 격차의 3.6배**다. 즉 우리가 따라잡아야 할 부분은 "오차를 평균적으로 더 줄이는 것"보다 "오차율 6%/8% 구간(계단 함수 경계)을 더 많은 시간대에서 넘기는 것"에 가깝다.

**참고 — 앞서 참고했던 1등 스냅샷과 지금 1등은 다른 사람이다**: 이전에 참고했던 스냅샷(Score 0.66961, 1-NMAE 0.89046, FICR 0.44876)과 지금 확인한 1등(Score 0.66993, 1-NMAE 0.8808, FICR 0.45905)은 **같은 사람이 전략을 바꾼 게 아니라 순위가 갈린 두 명의 서로 다른 결과**다. 그래서 "1등이 NMAE를 희생하고 FICR을 올리는 쪽으로 움직였다"는 식의 인과관계 해석은 근거가 없다 — 정정한다. 다만 두 스냅샷 모두 **비슷한 총점을 서로 다른 (1-NMAE, FICR) 조합으로 만들어낸다**는 사실 자체는 남는다: 상위권에는 "NMAE를 더 낮추는 전략"과 "FICR을 더 높이는 전략"이 둘 다 비슷한 총점에 도달할 수 있는 여지가 있다는 뜻으로, 우리에게 특정 방향(예: FICR 집중)이 유일한 정답이라고 단정할 근거는 아니다.

**결론적으로 확실한 것은**: 우리 결과와 지금 1등 사이의 격차 자체(FICR 격차가 NMAE 격차의 3.6배)뿐이다. 이것만으로 "FICR을 더 파야 한다"는 우선순위 판단은 유효하지만, 그 근거를 "1등의 전략 변화"에서 끌어오는 것은 잘못된 추론이었다.

**주의 — 이 비교는 어림값이다**: 우리 숫자는 2022~2024년 데이터로 만든 3-fold CV 평균이고, 리더보드 1등은 실제 2025년 테스트 데이터 예측 결과다. 기간·기상 조건이 다르므로 직접적인 "같은 시험 점수 비교"는 아니고, **어느 쪽 성분(NMAE vs FICR)이 상대적으로 더 벌어져 있는지 방향을 보는 용도**로만 쓴다. 위 셀을 실행해서 표를 확인한 뒤, 어느 성분 격차가 더 큰지에 따라 다음 우선순위(오차율 자체를 줄일지 vs 계단 구간을 넘기는 데 집중할지)를 정하자.

## 10. 오류 분석 — FICR 격차의 원인 찾기

**결론 먼저**: 9-8절에서 확인했듯 우리와 리더보드 1등의 격차는 FICR 쪽(0.075)이 1-NMAE 쪽(0.021)보다 3.6배 크다. `ensemble-final` 스킬 2절 체크리스트(풍속 구간별, 시간대/월별, 그룹별, **FICR 관점 — 오차율 6~8% 경계에 걸린 시간들의 공통점**)를 따라, 확정한 최종안(τ=0.70, actual 가중, 기본 하이퍼파라미터)의 검증 예측을 행 단위로 뜯어서 어떤 조건에서 6%/8% 경계를 못 넘는지 살펴본다.

**왜 앙상블보다 먼저 오류 분석인가**: `ensemble-final` 스킬 순서상 앙상블은 "분산을 줄여 극단 오차를 낮추는" 기법이라 일반적으로 도움이 되지만, 특정 조건(예: 특정 풍속 구간)에서 구조적으로 편향돼 있다면 그 원인을 먼저 알아야 한다 — 편향은 앙상블로 잘 안 없어지고(여러 모델이 같은 편향을 공유하기 쉬움), 원인 조건을 알면 그 구간만 피처 보강·구간별 보정 등으로 직접 대응할 수 있다.

**어떻게 오류 데이터를 만드는가**: 3개 fold 각각의 검증 구간(fold1=2023H2, fold2=2024H1, fold3=2024H2, 겹치지 않음)에 대해 그 fold의 학습 데이터로 학습한 모델의 예측을 모아 하나의 표로 합친다. 이렇게 하면 2023-07~2024-12 전체 기간을 "그 시점에 실제로 쓸 수 있었던 모델"의 예측으로 커버하는, OOF(out-of-fold — 학습에 안 쓰인 구간의 예측만 모은 것)와 비슷한 표가 된다. 다만 fold1/2/3은 서로 다른 학습 윈도우로 학습된 모델이라는 점은 감안한다(완전히 동일한 모델은 아님).

In [27]:
def build_oof_error_df(params, quantile_alpha=None, use_sample_weight=False, feature_cols=FEATURE_COLS, seed=SEED):
    features = feature_cols + ["group_id"]
    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    rows = []

    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        long_df = to_long_ext(fold_train, feature_cols)
        long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
        long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
        n_early = int(len(long_sorted) * 0.15)
        fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

        model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)
        fit_kwargs = {"sample_weight": fit_rows["actual_kwh"].to_numpy()} if use_sample_weight else {}
        model.fit(
            fit_rows[features], fit_rows["utilization"],
            eval_set=(early_rows[features], early_rows["utilization"]),
            cat_features=["group_id"], early_stopping_rounds=100, verbose=False, **fit_kwargs,
        )

        for g in TARGET_COLS:
            valid_g = fold_valid.copy()
            valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
            pred_kwh = np.clip(model.predict(valid_g[features]) * CAPACITY_KWH[g], 0, CAPACITY_KWH[g])
            mask = valid_g[g].notna().to_numpy()
            rows.append(pd.DataFrame({
                "fold": fold["name"],
                "forecast_kst_dtm": valid_g.loc[mask, "forecast_kst_dtm"].values,
                "group": g,
                "actual": valid_g.loc[mask, g].values,
                "pred": pred_kwh[mask],
                "ws10m": valid_g.loc[mask, f"{g}_ldaps_ws10m"].values,
            }))

    return pd.concat(rows, ignore_index=True)


error_df = build_oof_error_df(DEFAULT_PARAMS, quantile_alpha=0.70, use_sample_weight=True)
error_df["capacity"] = error_df["group"].map(CAPACITY_KWH)
error_df["error_rate"] = (error_df["actual"] - error_df["pred"]).abs() / error_df["capacity"]
error_df["is_scored"] = error_df["actual"] >= 0.10 * error_df["capacity"]
error_df["price_krw"] = np.select(
    [error_df["error_rate"] <= 0.06, error_df["error_rate"] <= 0.08], [4, 3], default=0,
)
error_df["hour"] = error_df["forecast_kst_dtm"].dt.hour
error_df["month"] = error_df["forecast_kst_dtm"].dt.month

scored_df = error_df[error_df["is_scored"]].copy()
print(f"전체 {len(error_df)}행 중 채점 대상(actual>=10%용량) {len(scored_df)}행 ({len(scored_df) / len(error_df):.1%})")
scored_df["error_rate"].describe()

전체 39577행 중 채점 대상(actual>=10%용량) 22160행 (56.0%)


count    22160.000000
mean         0.140082
std          0.122742
min          0.000003
25%          0.048916
50%          0.106032
75%          0.196989
max          0.836415
Name: error_rate, dtype: float64

### 10-1. 그룹별 오차율·경계 통과율

그룹마다 터빈 기종·위치·데이터 기간이 달라(`wind-domain-features` 스킬 7절) 오차 특성이 다를 수 있다. 그룹별로 평균 오차율과 "6% 이내 비율"/"8% 이내 비율"을 본다. 특히 라벨 기간이 짧은 group_3(2023-01부터)이 다른 그룹보다 불안정한지 확인한다.

In [28]:
group_summary = scored_df.groupby("group").agg(
    n=("error_rate", "size"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
    pct_within_8pct=("error_rate", lambda s: (s <= 0.08).mean()),
    mean_price_krw=("price_krw", "mean"),
)
group_summary

,n,mean_error_rate,pct_within_6pct,pct_within_8pct,mean_price_krw
group,,,,,
kpx_group_1,7642,0.132448,0.325308,0.423319,1.595263
kpx_group_2,7640,0.136749,0.324738,0.416754,1.575000
kpx_group_3,6878,0.152267,0.255743,0.340070,1.275952


### 10-2. 풍속 구간별 오차율 — 파워커브 급경사 구간 확인

`wind-domain-features` 스킬 1절: 파워커브는 cut-in(~3 m/s)과 rated(~11~13 m/s) 근처에서 급경사(풍속이 조금만 달라도 발전량이 크게 바뀜) 구간이라 오차가 커지기 쉽다. 여기서 쓰는 `ws10m`은 허브高(80~100 m)가 아니라 지상 10 m 풍속이라 절대적인 cut-in/rated 수치와는 다르지만(윈드시어로 허브 높이가 더 빠름), **상대적인 저/중/고풍속 구간 비교**에는 쓸 수 있다. 그룹별 표본 수를 맞추기 위해 그룹 전체를 합쳐 5분위(quintile)로 나눈다.

In [29]:
scored_df["ws_bin"] = pd.qcut(scored_df["ws10m"], 5)

ws_summary = scored_df.groupby("ws_bin", observed=True).agg(
    n=("error_rate", "size"),
    mean_ws10m=("ws10m", "mean"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
    pct_within_8pct=("error_rate", lambda s: (s <= 0.08).mean()),
)
ws_summary

,n,mean_ws10m,mean_error_rate,pct_within_6pct,pct_within_8pct
ws_bin,,,,,
"(0.102, 4.222]",4432,3.316209,0.120692,0.355370,0.465930
"(4.222, 5.371]",4432,4.806191,0.130137,0.308664,0.396435
"(5.371, 6.582]",4432,5.959518,0.151734,0.266020,0.351986
"(6.582, 8.294]",4432,7.342850,0.161107,0.260605,0.338448
"(8.294, 17.36]",4432,10.212900,0.136742,0.326940,0.423285


### 10-3. 시간대별·월별 오차율

산악풍은 주야 순환(낮/밤 대류 차이)과 계절 패턴이 뚜렷하다(`wind-domain-features` 스킬 5절). 시간대(0~23시)와 월(1~12월)별로 평균 오차율과 6% 이내 비율을 본다.

In [30]:
hour_summary = scored_df.groupby("hour").agg(
    n=("error_rate", "size"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
)

month_summary = scored_df.groupby("month").agg(
    n=("error_rate", "size"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
)

print("시간대별:")
print(hour_summary)
print("\n월별:")
print(month_summary)

시간대별:
         n  mean_error_rate  pct_within_6pct
hour                                        
0     1058         0.143817         0.311909
1     1062         0.134234         0.317326
2     1056         0.138825         0.320076
3     1062         0.136292         0.314501
4     1094         0.134809         0.319927
5     1052         0.135770         0.337452
6     1043         0.149957         0.295302
7      996         0.154175         0.289157
8      971         0.142644         0.285273
9      895         0.137593         0.292737
10     826         0.134807         0.305085
11     788         0.131513         0.309645
12     753         0.124624         0.310757
13     747         0.124072         0.344043
14     746         0.137964         0.294906
15     759         0.129800         0.330698
16     760         0.139130         0.319737
17     775         0.147602         0.280000
18     827         0.139761         0.285369
19     872         0.138791         0.283257
20  

### 10-4. 경계 근처(6%/8%) 시간대의 공통점 — FICR에 가장 직접적인 분석

`ensemble-final` 스킬 2절이 명시한 FICR 관점 체크: "오차율 6~8% 경계에 걸린 시간들의 공통점 (어떤 조건에서 아깝게 이탈하는가)". 오차율이 6%/8% 바로 위(±1%p)에 있는 시간대만 뽑아서, 그룹·풍속구간별로 어디에 몰려 있는지 본다. 여기 몰린 조건이 있다면 "그 구간만 오차를 1~2%p 더 줄이면 FICR이 즉시 오르는" 가장 효율 좋은 개선 지점이다.

In [31]:
near_6pct = scored_df[(scored_df["error_rate"] > 0.05) & (scored_df["error_rate"] <= 0.07)]
near_8pct = scored_df[(scored_df["error_rate"] > 0.07) & (scored_df["error_rate"] <= 0.09)]
over_8pct = scored_df[scored_df["error_rate"] > 0.08]

print(f"6% 경계 근처(5~7%) 시간대: {len(near_6pct)}건 ({len(near_6pct) / len(scored_df):.1%})")
print(near_6pct.groupby("group").size())
print(near_6pct.groupby("ws_bin", observed=True).size())

print(f"\n8% 경계 근처(7~9%) 시간대: {len(near_8pct)}건 ({len(near_8pct) / len(scored_df):.1%})")
print(near_8pct.groupby("group").size())
print(near_8pct.groupby("ws_bin", observed=True).size())

print(f"\n8% 초과(가격 0원 구간): {len(over_8pct)}건 ({len(over_8pct) / len(scored_df):.1%})")
print(over_8pct.groupby("group").size())
print(over_8pct.groupby("ws_bin", observed=True).size())

6% 경계 근처(5~7%) 시간대: 2087건 (9.4%)
group
kpx_group_1    761
kpx_group_2    746
kpx_group_3    580
dtype: int64
ws_bin
(0.102, 4.222]    491
(4.222, 5.371]    399
(5.371, 6.582]    412
(6.582, 8.294]    343
(8.294, 17.36]    442
dtype: int64

8% 경계 근처(7~9%) 시간대: 1961건 (8.8%)
group
kpx_group_1    712
kpx_group_2    669
kpx_group_3    580
dtype: int64
ws_bin
(0.102, 4.222]    468
(4.222, 5.371]    372
(5.371, 6.582]    352
(6.582, 8.294]    373
(8.294, 17.36]    396
dtype: int64

8% 초과(가격 0원 구간): 13402건 (60.5%)
group
kpx_group_1    4407
kpx_group_2    4456
kpx_group_3    4539
dtype: int64
ws_bin
(0.102, 4.222]    2367
(4.222, 5.371]    2675
(5.371, 6.582]    2872
(6.582, 8.294]    2932
(8.294, 17.36]    2556
dtype: int64


### 10-5. 종합 해석

**가장 먼저 봐야 할 숫자**: 채점 대상 22,160시간 중 **60.5%(13,402시간)가 오차율 8% 초과 — 가격이 0원**이다. 반면 6%/8% 경계 바로 근처(±1%p)에 있는 "아깝게 놓친" 시간대는 각각 9.4%/8.8%뿐이다. 즉 **FICR이 낮은 주된 이유는 "경계를 살짝 못 넘는 것"이 아니라 "애초에 오차가 크게 나는 시간이 절반 넘게 있는 것"**이다 — τ를 미세 조정하는 식의 보정(9절에서 이미 거의 다 짜냄)으로는 이 부분을 더 줄이기 어렵고, 예보 자체의 정확도(풍속 예보 오차, 그리고 그것이 v³로 증폭되는 구조)를 개선해야 하는 문제로 보인다.

**그룹별 — group_3이 뚜렷하게 약하다**: 8% 이내 비율이 group_1(42.3%)·group_2(41.7%) 대비 group_3은 34.0%로 낮고, **8% 초과(0원) 비율도 group_3이 66.0%로 group_1(57.7%)·group_2(58.3%)보다 눈에 띄게 높다.** `wind-domain-features` 스킬 7절 근거대로 group_3은 라벨 기간이 2023-01부터로 짧아(다른 그룹보다 학습에 쓸 수 있는 과거 데이터가 약 1년 적음) 그룹별 특성(파워커브, 최근접 격자)을 모델이 덜 배웠을 가능성이 높다.

**풍속 구간별 — 파워커브 이론과 정확히 들어맞는다**: 오차율이 가장 낮은 구간은 **가장 낮은 풍속 구간**(10m 풍속 0.1~4.2 m/s, 오차율 12.1%, 8%이내 46.6%)이고, 오차율이 가장 높은 구간은 **중간 풍속 구간**(10m 풍속 6.6~8.3 m/s, 오차율 16.1%, 8%이내 33.8%)이다. 가장 높은 풍속 구간(8.3~17.4 m/s)에서는 오차율이 다시 13.7%로 낮아진다. 이는 `wind-domain-features` 스킬 1절의 파워커브 이론과 정확히 일치한다 — 저풍속(컷인 근처)은 출력이 0에 가까워 맞히기 쉽고, 정격 근처로 올라가는 급경사 구간(우리 데이터의 10m 풍속으로는 중간 구간에 해당 — 윈드시어를 감안하면 허브高에서는 이 구간이 실제 정격 근접 풍속대일 가능성이 높다)은 풍속 오차가 발전량 오차로 크게 증폭되고, 정격 근처·이상(고풍속 구간)은 출력이 포화돼 있어 풍속 오차의 영향이 줄어든다.

**시간대·월별 — 부차적 신호**: 시간대는 큰 차이가 없지만 이른 아침(6~8시)·저녁(20~23시) 시간대가 조금 더 나쁘고 정오~오후 초반이 조금 낫다(대류 순환의 영향으로 추정). 월별로는 9월(오차율 10.8%, 6%이내 40.2%)이 가장 좋고 1월(16.0%, 27.7%)·3월(15.1%, 26.2%)·7월(15.3%, 25.3%)이 나쁘다 — 겨울철 변동성 큰 바람과 장마·태풍철(7월) 불안정성이 원인으로 추정되나, 뚜렷한 단조 패턴은 아니라 그룹·풍속 대비 우선순위는 낮다.

**다음 단계 우선순위 제안**:
1. **group_3 전용 대응**: 그룹별로 별도 τ 튜닝, 또는 group_3에 한해 표본 가중치를 더 주는 방식 검토
2. **중간 풍속 구간(급경사 구간) 정확도 개선**: 이 구간에 특화된 피처(풍속 변화율, gust 비율 등 이미 있는 피처들의 중요도를 이 구간에서만 따로 확인) 또는 구간별 보정항 검토
3. τ 미세조정으로 짜낼 수 있는 여지는 이미 9절에서 거의 소진됐다고 보고, 이 오류 분석 결과를 바탕으로 피처·구간별 접근에 시간을 쓰는 게 `model-tuning` 스킬 4절("튜닝으로 짜낸 0.001보다 피처 하나의 0.01이 크다")과 일치

## 요약

- `model-tuning`/`timeseries-validation` 스킬 기준, 확장 윈도우 3-fold CV로 바꿔 단일 fold 과적합 위험을 줄였다
- **8절 하이퍼파라미터 튜닝(Optuna 30 trial)**: 개선폭 +0.000045 — seed 표준편차보다 작아 "효과 미미"로 정직하게 기록
- **9절 손실 함수·가중치 조정**: actual 가중만으로는 미미(+0.0004)했지만, **분위수 회귀 τ=0.70 + actual 가중으로 +0.0305 개선**(seed 표준편차의 47.6배 — 안정적) — 이 노트북의 핵심 발견
- τ를 0.70~0.72까지 세밀 탐색해 진짜 peak(τ=0.71)를 확인했지만, 개선폭이 노이즈 수준이고 seed 표준편차가 3배 커서 **τ=0.70을 최종안으로 확정**
- 리더보드 1등(2026-07-22)과 Score를 1-NMAE/FICR로 분해해 비교: **FICR 격차(0.075)가 NMAE 격차(0.021)의 3.6배** — 다음 개선 우선순위는 평균 오차 축소보다 오차율 6%/8% 계단 경계를 넘기는 쪽
- **10절 오류 분석**: 채점 대상의 **60.5%가 오차율 8% 초과(0원)** — 경계 근처(±1%p)는 9% 남짓뿐이라 "미세 보정"보다 "예보 오차 자체 축소"가 문제. **group_3이 뚜렷하게 약함**(8% 초과 비율 66.0% vs group_1/2 57~58%), **중간 풍속 구간에서 오차 최대**(파워커브 급경사 구간 이론과 일치)

**최종 확정안**: 통합모델(CatBoost) + 기본 하이퍼파라미터(`DEFAULT_PARAMS`) + 분위수 회귀(τ=0.70) + actual 가중 학습. 8절에서 찾은 `BEST_PARAMS`(하이퍼파라미터 튜닝 결과)는 효과가 노이즈 수준이라 채택하지 않는다.

**다음 할 일**:
1. `train.ipynb`로 넘어가 τ=0.70 + actual 가중 설정을 하드코딩해 전체 데이터로 재학습 (`ensemble-final` 스킬 4~5절)
2. group_3 전용 대응(별도 τ 또는 표본 가중치)과 중간 풍속 구간 피처 보강은 시간 여유에 따라 재학습 이후 별도 실험으로 고려
3. `HANDOFF.md` 갱신 — 세션 종료 시 `session-handoff` 스킬 참조